# Préparation de Données 
Ce notebook est la version finale qui règle le problème des gares disparaissant à cause du format booléen.

- **OHE en Int** : Utilisation de `dtype=int` dans `get_dummies` pour transformer les gares en 1 et 0 (et non True/False).
- **Préservation des Features** : Le filtre numérique final conserve désormais toutes les gares encodées.
- **Suppression du Texte** : Élimination du texte brut des commentaires.
- **Gestion des Infinis** : Correction des divisions par zéro.

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- CONFIGURATION DES CHEMINS ---
base_path = Path(r'C:/Users/Utilisateur/Documents/Albert School/B2/SEMESTRE 2/Achieving a ML Proof of Concept/ml-poc-project')
input_file = base_path / 'data' / 'data_nettoyee.csv'
output_file = base_path / 'data' / 'processed_dataset.csv'

print(f'Chargement des données : {input_file}')

Chargement des données : C:\Users\Utilisateur\Documents\Albert School\B2\SEMESTRE 2\Achieving a ML Proof of Concept\ml-poc-project\data\data_nettoyee.csv


## 1. Chargement et Target
On calcule la cible et on traite les valeurs infinies.

In [10]:
df = pd.read_csv(input_file)
df['Date'] = pd.to_datetime(df['Date'])

# Cible
df['trains_ok'] = df['Nombre de circulations prévues'] - df['Nombre de trains annulés']
df['target'] = df["Nombre de trains en retard à l'arrivée"] / df['trains_ok']
df['target'] = df['target'].replace([np.inf, -np.inf], 0).fillna(0)

# ID technique
df['id_ligne'] = pd.factorize(df['Gare de départ'] + ' -> ' + df["Gare d'arrivée"])[0] + 1

print(f'Lignes chargées : {len(df)}')

Lignes chargées : 8874


## 2. Encodage (Correction 1/0)
- **Commentaire** : Transformation en indicateur binaire (0 ou 1).
- **Service & Gares** : One-Hot Encoding complet.

On transforme les gares en **entiers (0 et 1)** et non en booléens.

In [11]:
# 1. Commentaire binaire
col_comm = "Commentaire retards à l'arrivée"
if col_comm in df.columns:
    df['has_commentaire'] = df[col_comm].apply(
        lambda x: 1 if pd.notnull(x) and str(x).strip() not in ['', 'Aucun commentaire', 'nan'] else 0
    )
    df = df.drop(columns=[col_comm])

# 2. One-Hot Encoding avec DTYPE=INT (Fix gares manquantes)
cols_to_ohe = ['Service', 'Gare de départ', "Gare d'arrivée"]
df = pd.get_dummies(df, columns=[c for c in cols_to_ohe if c in df.columns], prefix=['ser', 'dep', 'arr'], dtype=int)

print(f'Colonnes après encodage : {len(df.columns)} (Les gares sont maintenant des 0/1)')

Colonnes après encodage : 146 (Les gares sont maintenant des 0/1)


## 3. Features Temporelles et Surcharge
Ajout des lags, moyennes mobiles et de la saisonnalité cyclique.

Dans une série temporelle, la valeur à l'instant $t$ dépend souvent des valeurs passées $t-1, t-2, ...$

- **Lags** : On décale la cible pour donner au modèle une vision du passé.
- **Moyenne Mobile** : On calcule la moyenne du retard sur les 3 derniers mois pour capturer la tendance.

Un mois est une variable circulaire. Pour que le modèle comprenne que Décembre (12) est proche de Janvier (1), on utilise :

$$x_{sin} = \sin\left(\frac{2\pi \cdot month}{12}\right)$$
$$x_{cos} = \cos\left(\frac{2\pi \cdot month}{12}\right)$$

On ajoute des variables metiers : 
- **Volume de trafic** : Surcharge du réseau.
- **Annulations** : Signal de crises majeures.
- **Grands Départs** : Flag pour Juillet, Août, Décembre.

In [12]:
df = df.sort_values(['id_ligne', 'Date'])

# Lags
for i in [1, 2, 3]:
    df[f'lag_{i}'] = df.groupby('id_ligne')['target'].shift(i)

# Saisonnalité
df['month'] = df['Date'].dt.month
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Surcharge
df['surcharge_absolue'] = df['Nombre de circulations prévues']
df['surcharge_relative'] = df.groupby('id_ligne')['Nombre de circulations prévues'].transform(lambda x: x / x.mean())
df['densite_trafic'] = df['Nombre de circulations prévues'] / df['Durée moyenne du trajet']
df['densite_trafic'] = df['densite_trafic'].replace([np.inf, -np.inf], 0)

print('Features temporelles et surcharge terminées.')

Features temporelles et surcharge terminées.


# 4. Retrait des outlieurs

On enlève du dataset les données pour lesquelles le taux de retard est supérieur à 0.5 car elles sont très peu et on veut se concentrer sur les petits retards biens predis 

In [13]:
#Retrait outliers taux de retard >0.5

df = df.drop(df[df['taux_retard'] > 0.5].index)

## 4. Sauvegarde Finale
On garde la Date et TOUS les chiffres (incluant les gares encodées).

In [14]:
# Nettoyage NaN des lags
df_final = df.dropna(subset=['lag_3']).copy()

# Conservation des colonnes numériques uniquement (plus la Date)
numeric_cols = df_final.select_dtypes(include=[np.number]).columns.tolist()
df_final = df_final[['Date'] + numeric_cols]

# Sauvegarde
df_final.to_csv(output_file, index=False)

print(f'✅ SUCCÈS : Fichier sauvegardé dans {output_file}')
print(f'Vérification : La colonne dep_PARIS_LYON (ou autre gare) existe bien.')
print(df_final)

✅ SUCCÈS : Fichier sauvegardé dans C:\Users\Utilisateur\Documents\Albert School\B2\SEMESTRE 2\Achieving a ML Proof of Concept\ml-poc-project\data\processed_dataset.csv
Vérification : La colonne dep_PARIS_LYON (ou autre gare) existe bien.
           Date  Durée moyenne du trajet  Nombre de circulations prévues  \
453  2018-04-01                      141                             841   
521  2018-05-01                      140                             844   
703  2018-06-01                      142                             844   
780  2018-07-01                      146                             817   
910  2018-08-01                      146                             738   
...         ...                      ...                             ...   
8278 2023-08-01                      191                             224   
8494 2023-09-01                      191                             237   
8574 2023-10-01                      192                             244   
86